# Bradford Bulls — Team-Aware Frame Extraction v2.0

### Improvements over v1.0
- **Auto-Calibration**: K-Means clustering on torso colors → works with ANY jersey color
- **Overlay Masking**: Detects scoreboard/watermark → excludes from sharpness scoring
- **Team-Aware Scoring**: Prioritizes frames with YOUR team, not just any person
- **Smart Pitch Filter**: Bypasses green-check for close-ups (most valuable frames)
- **Foreground Filter**: Separates players from crowd in background
- **Quota Selection**: Balanced dataset — 65% target / 20% mixed / 15% opponent

### Pipeline
```
Video (Google Drive)
  → Phase 0A: Detect static overlays (scoreboard, watermark)
  → Phase 0B: Auto-calibrate team colors (K-Means + user picks team)
  → Pass 1: Fast scan → find quality segments
  → Pass 2: Team-aware extraction (classify each player's team)
  → Quota selection (65% target / 20% mixed / 15% opponent)
  → Save frames + enriched metadata CSV
```

## 0. Install Dependencies
Run this cell, then **Restart Runtime**.

In [ ]:
!pip install -q ultralytics>=8.3.0 scikit-learn imagehash scikit-image pillow opencv-python-headless roboflow
print("\nDone. Please RESTART RUNTIME now (Runtime > Restart runtime).")

## 1. YOUR CONFIG (Only cell you edit)

In [ ]:
# ============================================================
# YOUR CONFIG — Edit ONLY these lines
# ============================================================

MEMBER_NAME = "your_name"                    # e.g. "hoa"
VIDEO_FILENAME = "M01_white_1080p.mp4"       # Exact filename on Drive
TEST_MODE = True                             # True=test 2000 frames, False=full

# ============================================================
# DO NOT EDIT BELOW THIS LINE
# ============================================================

## 2. Setup Environment

In [ ]:
import torch, cv2, os, sys, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
from ultralytics import YOLO

# --- GPU Check ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)")
else:
    print("WARNING: No GPU detected.")

# --- Mount Google Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Directory Structure ---
DRIVE_ROOT = Path("/content/drive/MyDrive/Bradford_Bulls")
VIDEOS_DIR = DRIVE_ROOT / "videos"
FRAMES_DIR = DRIVE_ROOT / "frames"
METADATA_DIR = DRIVE_ROOT / "metadata"
REPORTS_DIR = DRIVE_ROOT / "reports"
SRC_DIR = DRIVE_ROOT / "src"  # Python modules on Drive

for d in [VIDEOS_DIR, FRAMES_DIR, METADATA_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Add src to Python path ---
# Option 1: from Drive (upload src/ folder to Bradford_Bulls/)
if (SRC_DIR / "frame_extraction").exists():
    sys.path.insert(0, str(SRC_DIR))
    print(f"Using modules from Drive: {SRC_DIR}")
# Option 2: from cloned repo
elif Path("/content/BRADFORD_BULLS_PROJECT/src").exists():
    sys.path.insert(0, "/content/BRADFORD_BULLS_PROJECT/src")
    print("Using modules from cloned repo")
else:
    print("ERROR: Cannot find src/frame_extraction/ modules.")
    print("Please either:")
    print("  1. Upload src/ folder to Bradford_Bulls/ on Drive")
    print("  2. Or clone the repo: !git clone <repo-url> /content/BRADFORD_BULLS_PROJECT")
    raise FileNotFoundError("src/frame_extraction not found")

from frame_extraction.overlay import detect_static_overlays, visualize_overlay
from frame_extraction.calibration import auto_calibrate
from frame_extraction.pipeline import pass1_fast_scan, pass2_extract, DEFAULT_PARAMS
from frame_extraction.selection import select_by_quota, print_selection_summary
from frame_extraction.helpers import fmt_timestamp

# --- Validate video ---
MATCH_ID = Path(VIDEO_FILENAME).stem
VIDEO_PATH = VIDEOS_DIR / VIDEO_FILENAME
FRAMES_OUTPUT_DIR = FRAMES_DIR / MATCH_ID
FRAMES_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not VIDEO_PATH.exists():
    available = list(VIDEOS_DIR.glob("*.mp4"))
    print(f"ERROR: '{VIDEO_FILENAME}' not found in {VIDEOS_DIR}")
    for v in available:
        print(f"  - {v.name} ({v.stat().st_size/1e6:.0f} MB)")
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")

print(f"\nVideo: {VIDEO_PATH.name} ({VIDEO_PATH.stat().st_size/1e6:.0f} MB)")
print(f"Match ID: {MATCH_ID}")
print(f"Output: {FRAMES_OUTPUT_DIR}")

## 3. Load Video & Model

In [ ]:
# --- Video metadata ---
cap = cv2.VideoCapture(str(VIDEO_PATH))
FPS = cap.get(cv2.CAP_PROP_FPS)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
DURATION_SEC = TOTAL_FRAMES / FPS
cap.release()

# --- Parameters ---
PERSON_DETECTION_MODEL = "yolo11l.pt"  # Large model for best accuracy
JPEG_QUALITY = 95

if TEST_MODE:
    MAX_SCAN_FRAMES = 2000
    TARGET_FRAMES = 40
    print(f"TEST MODE: scanning {MAX_SCAN_FRAMES} frames, selecting ~{TARGET_FRAMES}")
else:
    MAX_SCAN_FRAMES = TOTAL_FRAMES
    TARGET_FRAMES = 350  # Adjust based on video length
    print(f"PRODUCTION MODE: scanning all {TOTAL_FRAMES:,} frames, selecting ~{TARGET_FRAMES}")

print(f"\n{'='*60}")
print(f"  VIDEO INFO")
print(f"{'='*60}")
print(f"  Duration:    {DURATION_SEC/60:.1f} min | Resolution: {W}x{H} | FPS: {FPS:.0f}")
print(f"  Total:       {TOTAL_FRAMES:,} frames")
print(f"{'='*60}")

# --- Load YOLO model ---
print(f"\nLoading {PERSON_DETECTION_MODEL}...")
yolo_model = YOLO(PERSON_DETECTION_MODEL)
print(f"Model loaded on {DEVICE}")

## 4. Phase 0A — Detect Static Overlays

Finds scoreboard, watermark, channel logo by analyzing
which pixels stay constant across the entire video.

In [ ]:
print("Detecting static overlays (scoreboard, watermark)...")
overlay_mask, overlay_ratio = detect_static_overlays(VIDEO_PATH)
print(f"Overlay coverage: {overlay_ratio*100:.1f}% of frame masked")

# Visualize on a sample frame
cap = cv2.VideoCapture(str(VIDEO_PATH))
cap.set(cv2.CAP_PROP_POS_FRAMES, TOTAL_FRAMES // 3)
ret, sample_frame = cap.read()
cap.release()
if ret:
    visualize_overlay(sample_frame, overlay_mask)

## 5. Phase 0B — Auto-Calibrate Team Colors

Clusters player jersey colors automatically.
You just need to **pick which cluster is your team** (enter one number).

In [ ]:
calibration = auto_calibrate(
    VIDEO_PATH, yolo_model, DEVICE,
    overlay_mask=overlay_mask,
    n_sample_frames=60,
)

if calibration is None:
    print("\nWARNING: Calibration failed. Pipeline will run without team classification.")
    print("All frames will be treated equally (no team-based scoring).")

## 6. Pass 1 — Fast Scan (Zoom Timeline)

Scans every 5th frame to find when the camera is zoomed into players.

In [ ]:
segments, zoom_timeline, video_info = pass1_fast_scan(
    VIDEO_PATH, yolo_model, DEVICE, max_frames=MAX_SCAN_FRAMES
)

total_seg_frames = sum(s[-1][0] - s[0][0] + 5 for s in segments)
print(f"\n{'='*60}")
print(f"  PASS 1 RESULTS — {MATCH_ID}")
print(f"{'='*60}")
print(f"  Scanned:          {len(zoom_timeline):,} frames")
print(f"  Quality segments: {len(segments)}")
print(f"  Segment coverage: {total_seg_frames/max(MAX_SCAN_FRAMES,1)*100:.1f}%")
if segments:
    avg_len = np.mean([(s[-1][0]-s[0][0])/FPS for s in segments])
    print(f"  Avg segment len:  {avg_len:.1f}s")
print(f"{'='*60}")

## 7. Pass 2 — Team-Aware Extraction

For each quality segment: detect players → classify team → score → pick best.

In [ ]:
candidates, pass2_stats = pass2_extract(
    VIDEO_PATH, segments, yolo_model, DEVICE,
    calibration, overlay_mask, video_info,
    max_frames=MAX_SCAN_FRAMES
)

print(f"\n{'='*60}")
print(f"  PASS 2 RESULTS — {MATCH_ID}")
print(f"{'='*60}")
print(f"  Segments processed:  {pass2_stats['segments_processed']}")
print(f"  Frames analyzed:     {pass2_stats['frames_analyzed']:,}")
print(f"  Skipped (no pitch):  {pass2_stats['skipped_pitch']:,}")
print(f"  Skipped (blurry):    {pass2_stats['skipped_blurry']:,}")
print(f"  Total candidates:    {len(candidates)}")
print(f"    Target team:       {pass2_stats['team_target']}")
print(f"    Mixed:             {pass2_stats['team_mixed']}")
print(f"    Opponent:          {pass2_stats['team_opponent']}")
print(f"{'='*60}")

## 8. Quota Selection

Select balanced dataset: 65% target / 20% mixed / 15% opponent.

In [ ]:
selected, sel_stats = select_by_quota(candidates, TARGET_FRAMES)
print_selection_summary(selected, sel_stats, len(candidates))

## 9. Save Frames & Metadata

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
rows = []
saved = 0

for idx, meta in enumerate(tqdm(selected, desc="Saving frames")):
    cap.set(cv2.CAP_PROP_POS_FRAMES, meta["frame_num"])
    ret, frame = cap.read()
    if not ret:
        continue

    fname = f"{MATCH_ID}_{meta['frame_num']:06d}_{meta['timestamp_hms']}.jpg"
    fpath = FRAMES_OUTPUT_DIR / fname
    cv2.imwrite(str(fpath), frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
    saved += 1

    rows.append({
        "id": idx + 1, "filename": fname,
        "frame_num": meta["frame_num"],
        "timestamp_sec": meta["timestamp_sec"],
        "timestamp_hms": meta["timestamp_hms"],
        "category": meta["category"],
        "shot_type": meta["shot_type"],
        "n_persons": meta["n_persons"],
        "n_target": meta["n_target"],
        "n_opponent": meta["n_opponent"],
        "team_dominance": meta["team_dominance"],
        "sharpness": meta["sharpness"],
        "is_motion_blur": meta["is_motion_blur"],
        "max_person_area_ratio": meta["max_person_area_ratio"],
        "max_target_area_ratio": meta["max_target_area_ratio"],
        "target_coverage": meta["target_coverage"],
        "pitch_green_ratio": meta["pitch_green_ratio"],
        "score": meta["score"],
        "segment_idx": meta["segment_idx"],
        "match_id": MATCH_ID,
        "member": MEMBER_NAME,
        "extracted_at": datetime.now().isoformat(),
    })

cap.release()

df = pd.DataFrame(rows)
csv_path = METADATA_DIR / f"{MATCH_ID}_v2_index.csv"
df.to_csv(csv_path, index=False)

print(f"\nSaved {saved} frames to: {FRAMES_OUTPUT_DIR}")
print(f"Metadata: {csv_path}")

## 10. Preview by Category

In [ ]:
if len(df) == 0:
    print("No frames saved.")
else:
    # Show top frames per category
    categories = df["category"].unique()
    for cat in sorted(categories):
        cat_df = df[df["category"] == cat].sort_values("score", ascending=False)
        n_show = min(4, len(cat_df))
        if n_show == 0:
            continue

        fig, axes = plt.subplots(1, n_show, figsize=(5*n_show, 5))
        if n_show == 1:
            axes = [axes]

        for i, (_, row) in enumerate(cat_df.head(n_show).iterrows()):
            fpath = FRAMES_OUTPUT_DIR / row["filename"]
            if fpath.exists():
                img = cv2.imread(str(fpath))
                axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                axes[i].set_title(
                    f"{row['timestamp_hms']} | score={row['score']:.2f}\n"
                    f"target={row['n_target']} opp={row['n_opponent']} | {row['shot_type']}",
                    fontsize=8
                )
            axes[i].axis("off")

        plt.suptitle(f"{cat.upper()} — {len(cat_df)} frames", fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.show()

    # Summary
    print(f"\nCategory distribution:")
    print(df["category"].value_counts().to_string())
    print(f"\nShot type distribution:")
    print(df["shot_type"].value_counts().to_string())
    print(f"\nTotal frames on Drive: {len(list(FRAMES_OUTPUT_DIR.glob(f'{MATCH_ID}_*.jpg')))}")

## 11. Quality Distribution

In [ ]:
if len(df) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    axes[0,0].hist(df["sharpness"], bins=25, color="steelblue", edgecolor="white")
    axes[0,0].set_title("Sharpness Distribution")

    axes[0,1].hist(df["score"], bins=25, color="forestgreen", edgecolor="white")
    axes[0,1].set_title("Quality Score")

    axes[0,2].hist(df["team_dominance"], bins=20, color="coral", edgecolor="white")
    axes[0,2].set_title("Team Dominance (target ratio)")

    axes[1,0].hist(df["max_person_area_ratio"], bins=25, color="mediumpurple", edgecolor="white")
    axes[1,0].set_title("Largest Person Size (% frame)")

    axes[1,1].hist(df["timestamp_sec"]/60, bins=25, color="goldenrod", edgecolor="white")
    axes[1,1].set_title("Temporal Distribution (minutes)")

    cat_counts = df["category"].value_counts()
    axes[1,2].barh(cat_counts.index, cat_counts.values, color="teal", edgecolor="white")
    axes[1,2].set_title("Frames per Category")

    plt.suptitle(f"Quality Metrics — {MATCH_ID} ({len(df)} frames)", fontsize=14)
    plt.tight_layout()
    plt.show()

## 12. Upload to Roboflow (Optional)

In [ ]:
import getpass
from roboflow import Roboflow

ROBOFLOW_API_KEY = getpass.getpass("Paste Roboflow API Key: ")
ROBOFLOW_PROJECT = "kit-sponsor-logos"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
workspace = rf.workspace()
project = workspace.project(ROBOFLOW_PROJECT)
print(f"Connected to: {project.name}")

frames_to_upload = sorted(FRAMES_OUTPUT_DIR.glob(f"{MATCH_ID}_*.jpg"))
print(f"Uploading {len(frames_to_upload)} frames...")

tag_str = f"{MATCH_ID},{MEMBER_NAME},v2"
success, errors = 0, 0
for fp in tqdm(frames_to_upload, desc="Uploading"):
    try:
        project.upload(image_path=str(fp), split="train", tag_names=[tag_str])
        success += 1
    except Exception as e:
        errors += 1
        if errors <= 3:
            print(f"  Error: {e}")

print(f"\nDone: {success} uploaded, {errors} errors")

## 13. Checklist

- [ ] `TEST_MODE = False` for production run
- [ ] Calibration: verified correct team cluster
- [ ] Overlay mask: scoreboard/watermark detected
- [ ] Preview: target team frames have visible logos
- [ ] Category mix: ~65% target / ~20% mixed / ~15% opponent
- [ ] Metadata CSV saved with team breakdown columns
- [ ] Uploaded to Roboflow with v2 tag